# 출고 데이터 분석 시작하기

환경 확인 → 파일 목록 → 앞부분 표본 → 수량 분포 순서로 실행합니다. 원본 파일은 수정하지 않습니다.

In [ ]:
import sys

import duckdb
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display
from matplotlib import font_manager

from hit_stocks_analysis import find_project_root, list_outbound_files, load_sample

ROOT = find_project_root()
sns.set_theme(style="whitegrid")
available_fonts = {font.name for font in font_manager.fontManager.ttflist}
for font in ("AppleGothic", "Malgun Gothic", "NanumGothic"):
    if font in available_fonts:
        plt.rcParams["font.family"] = font
        break
plt.rcParams["axes.unicode_minus"] = False
print("Python:", sys.version.split()[0])
print("프로젝트:", ROOT)
print("pandas:", pd.__version__, "DuckDB:", duckdb.__version__)

## 원본 파일 목록

In [ ]:
files = list_outbound_files()
file_summary = pd.DataFrame(
    {"file": path.name, "size_mb": round(path.stat().st_size / 1_000_000, 1)} for path in files
)
display(file_summary)

## 첫 파일의 앞 10,000행

전체 기간의 대표 표본이 아닙니다. 주문번호·SKU·위치 코드는 문자열로 보존합니다.

In [ ]:
sample = load_sample(files[0], nrows=10_000)
display(sample.head())
print("표본 크기:", sample.shape)
display(pd.DataFrame({"dtype": sample.dtypes.astype(str), "missing": sample.isna().sum()}))

## 표본 수량 분포

변환할 수 없는 수량은 결측치로 처리하고 개수를 표시합니다.

In [ ]:
quantity_columns = ["outboundRequestQuantity", "pickingQuantity", "shortageQuantity"]
quantities = sample[quantity_columns].apply(pd.to_numeric, errors="coerce")
display(quantities.describe())
print("수량 결측/변환 실패:", quantities.isna().sum().to_dict())
fig, ax = plt.subplots(figsize=(10, 4))
sns.histplot(quantities["outboundRequestQuantity"].dropna(), bins=40, ax=ax)
ax.set(title="Requested quantity — first-file sample", xlabel="Quantity", ylabel="Rows")
plt.show()

## DuckDB로 전체 파일 조회하기

아래 셀은 모든 CSV를 조회 대상으로 등록한 뒤 5행만 가져옵니다. 모든 열을 문자열로 읽어 파일 간 타입 추론 차이와 식별자 변환을 방지합니다. 전체 행 수·합계는 다음 SQL 예제를 별도 셀에서 실행할 수 있습니다.

```python
con.sql("SELECT COUNT(*) AS total_rows FROM outbound").df()
```

수량을 집계할 때는 `TRY_CAST(outboundRequestQuantity AS DOUBLE)`처럼 명시적으로 변환하고, 변환 실패 및 결측치 수를 확인하세요. 전체 조회는 파일 크기에 따라 시간이 걸립니다.

In [ ]:
con = duckdb.connect(":memory:")
con.read_csv(
    [str(path) for path in files],
    header=True,
    all_varchar=True,
    union_by_name=True,
).create_view("outbound")
display(con.sql("SELECT * FROM outbound LIMIT 5").df())

## 결과 저장 예시

필요할 때 아래 명령을 별도 셀에서 실행하세요.

```python
sample.to_parquet(ROOT / "data" / "outbound_sample.parquet", index=False)
fig.savefig(ROOT / "reports" / "requested_quantity.png", dpi=150, bbox_inches="tight")
```

분석이 끝나면 아래 셀로 DuckDB 연결을 닫습니다. 다시 SQL을 실행하려면 연결 생성 셀부터 실행하세요.

In [ ]:
con.close()